# Notebook 01 — Pipeline A: Reproducing the Dataset Authors' Preprocessing

**Project:** Reproducible comparison of EEG preprocessing pipelines on OpenNeuro `ds004504`
(the "AHEPA" resting-state EEG dataset: Alzheimer's disease, frontotemporal dementia, controls).

---

## What this notebook does

1. Verifies the dataset against its primary sources (it does **not** trust the numbers quoted in the project brief).
2. Builds a dataset inventory and runs integrity checks.
3. Implements **Pipeline A** — our reproduction of the authors' published preprocessing — *from the raw EEG*.
4. Validates that reproduction against the authors' own supplied derivative files.
5. Runs the shared downstream analysis: epoching → features → subject-level cross-validation.
6. Measures signal quality and computational cost.

## What this notebook deliberately does **not** do

- It does not simply load the authors' cleaned files and call that a reproduction. Pipeline A starts from raw.
- It does not fabricate any number. Every table is populated by executing code; anything that fails is logged and reported as an exclusion.

## Run order

`01_author_pipeline` → `02_alternative_pipeline` → `03_pipeline_comparison`

> **Before you start:** upload `ds004504_common.py` to the folder named in `MODULE_DIR` below.

## 1. Environment setup

### 1.1 Install packages

`asrpy` provides Artifact Subspace Reconstruction (the Python equivalent of EEGLAB's `clean_rawdata`), and `mne-icalabel` provides the ICLabel classifier. Both are required for a faithful Pipeline A.

In [ ]:
# Colab package installation.
# Versions are pinned loosely (>=) so the notebook keeps working as Colab's base image
# moves, but every resolved version is RECORDED later by get_environment_info().
%pip install -q "mne>=1.6" "scikit-learn>=1.3" "pandas>=2.0" "scipy>=1.10" \
                "matplotlib>=3.7" "pyarrow>=12.0" "psutil>=5.9" \
                "openneuro-py>=2024.1" "asrpy>=0.0.3" "mne-icalabel>=0.6" "onnxruntime>=1.16"

print("Installation finished. If Colab asks you to restart the runtime, do so and "
      "then re-run from this cell onwards.")

### 1.2 Mount Google Drive

In [ ]:
# Google Drive is OPTIONAL. It is worth using because the cache and results survive a
# Colab session timeout -- without it, a disconnect means reprocessing from scratch.
# Set USE_DRIVE = False to run entirely on the ephemeral Colab disk (or locally).
# The DATASET itself is always downloaded at runtime and never needs to be in Drive.

USE_DRIVE = True          # <-- set False to skip Drive entirely

IN_COLAB = False
DRIVE_OK = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = True
        print("Drive mounted: results and cache will persist across sessions.")
    except Exception as exc:
        print(f"Drive mount failed ({exc.__class__.__name__}): {exc}")
        print("Continuing on the local Colab disk. Results will be LOST on disconnect "
              "unless you download them before the session ends.")
elif IN_COLAB:
    print("USE_DRIVE = False. Working on the local Colab disk; "
          "download your results before the session ends.")
else:
    print("Not running in Colab. Using local directories.")

### 1.3 Load the shared analysis module

Every function that touches features, cross-validation or statistics lives in one module that **both** pipeline notebooks import. This is a structural guarantee that the downstream analysis is identical, rather than a promise that it is.

In [ ]:
# `ds004504_common.py` holds every pipeline, feature, cross-validation and statistics
# function. Both preprocessing notebooks import the SAME module, which is how the
# "fair comparison" requirement is enforced structurally rather than by convention.
#
# This cell looks for the module in the usual places and, if it cannot find it,
# opens Colab's file picker so you can upload it directly -- no Drive required.
import sys
from pathlib import Path

CANDIDATE_DIRS = [
    "/content/drive/MyDrive/ds004504_experiment",   # Drive, if mounted
    "/content",                                      # Colab working dir
    ".",                                             # local / cwd
]

def _locate_module():
    for d in CANDIDATE_DIRS:
        if (Path(d) / "ds004504_common.py").exists():
            return d
    return None

MODULE_DIR = _locate_module()

if MODULE_DIR is None and IN_COLAB:
    print("ds004504_common.py not found. Please upload it now.")
    try:
        from google.colab import files
        files.upload()
        MODULE_DIR = _locate_module()
    except Exception as exc:
        print(f"Upload failed: {exc!r}")

if MODULE_DIR is None:
    raise FileNotFoundError(
        "ds004504_common.py could not be found in any of: "
        + ", ".join(CANDIDATE_DIRS)
        + ". Upload it next to this notebook (or into Drive) and re-run."
    )

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import ds004504_common as ds
print(f"Loaded ds004504_common v{ds.MODULE_VERSION} from {MODULE_DIR}")

### 1.4 Standard imports

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("ERROR")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "figure.facecolor": "white"})
print("Imports ready.")

### 1.5 Configuration

All tunable parameters live here and are serialised to `results/config.json` so that any result can be traced back to the exact settings that produced it.

In [ ]:
# =====================================================================================
# CONFIGURATION -- the single place to change how this notebook runs
# =====================================================================================
# RUN MODES (identical code path, only MAX_SUBJECTS changes):
#     Quick test  : MAX_SUBJECTS = 5     (smoke test; too few for real statistics)
#     Development : MAX_SUBJECTS = 20
#     Full        : MAX_SUBJECTS = None  (all 88 subjects)
# Start with 5 to check everything runs, then raise it.

DATASET_PATH = "/content/ds004504"
OUTPUT_PATH  = ("/content/drive/MyDrive/ds004504_experiment" if DRIVE_OK
                else "/content/ds004504_experiment" if IN_COLAB
                else "./ds004504_experiment")
MAX_SUBJECTS = 20            # <-- change this to switch run mode
RANDOM_SEED  = 42
N_JOBS       = 1             # Colab free tier has ~2 vCPUs; 1 keeps peak RAM predictable

cfg = ds.Config(
    dataset_path=DATASET_PATH,
    output_path=OUTPUT_PATH,
    max_subjects=MAX_SUBJECTS,
    random_seed=RANDOM_SEED,
    n_jobs=N_JOBS,
    # --- preprocessing parameters shared by BOTH pipelines (controlled variables) ---
    l_freq=0.5, h_freq=45.0, butter_order=4,
    resample_to=100.0,       # 500 -> 100 Hz; safe below the 45 Hz low-pass, 5x cheaper
    reference="average",     # see ds.NOTE_ON_REFERENCE for the A1-A2 caveat
    # --- epoching / features (identical for both pipelines) ---
    epoch_length_s=4.0, epoch_overlap_s=0.0,
    # --- cross-validation ---
    cv_n_splits=10, cv_n_repeats=5, run_loso=True,
    n_bootstrap=10000,
    cache_enabled=True, overwrite_cache=False,
)
cfg.make_dirs()

print(f"Run mode      : {cfg.mode()}")
print(f"Max subjects  : {cfg.max_subjects}")
print(f"Output path   : {cfg.out}")
print(f"Cache path    : {cfg.cache_dir}")
saved = cfg.save()
print(f"Config saved  : {saved}")

### 1.6 Record the execution environment

Runtime numbers are only comparable within a single hardware environment. Capturing it is what makes the later computational benchmark interpretable.

In [ ]:
# Capture the exact execution environment. Runtime comparisons are only meaningful
# within one environment, so this record is what lets a reader judge our timings.
import json
env = ds.get_environment_info()
ds.save_results(cfg, "environment", env)

print(f"Python      : {env['python_version']}")
print(f"CPU         : {env.get('cpu_model', 'unknown')}")
print(f"Logical CPUs: {env['cpu_count_logical']}")
print(f"RAM total   : {env['ram_total_gb']} GB")
print(f"GPU         : {env['gpu']}  (not used -- this experiment is CPU-only by design)")
print("\nKey package versions:")
for pkg in ["mne", "numpy", "scipy", "scikit-learn", "sklearn", "asrpy", "mne_icalabel"]:
    if pkg in env["packages"]:
        print(f"  {pkg:<15} {env['packages'][pkg]}")

### 1.7 Logging

Every warning and per-subject failure is written to a log file, so nothing disappears silently.

In [ ]:
log_path = ds.setup_logging(cfg, name="pipeline_A")
print("Logging to:", log_path)

---
## 2. The authors' preprocessing pipeline — verified against primary sources

### 2.1 Source of the pipeline description

The preprocessing description is quoted verbatim in the dataset's own `README`
(OpenNeuro `ds004504`, mirrored at `github.com/OpenNeuroDatasets/ds004504`), and the
dataset is documented in the peer-reviewed data descriptor:

> Miltiadous, A., Tzimourta, K. D., Afrantou, T., Ioannidis, P., Grigoriadis, N.,
> Tsalikakis, D. G., Angelidis, P., Tsipouras, M. G., Glavas, E., Giannakeas, N., &
> Tzallas, A. T. (2023). *A Dataset of Scalp EEG Recordings of Alzheimer's Disease,
> Frontotemporal Dementia and Healthy Subjects from Routine EEG.* **Data**, 8(6), 95.
> https://doi.org/10.3390/data8060095

The README states the pipeline as: a Butterworth band-pass filter of 0.5–45 Hz,
re-referencing to A1–A2, then Artifact Subspace Reconstruction removing bad data periods
exceeding a maximum acceptable 0.5-second-window standard deviation of 17, then ICA using
the RunICA algorithm producing 19 components, from which components classified as eye or
jaw artifacts by EEGLAB's ICLabel routine were automatically rejected.

### 2.2 Acquisition parameters (from the same source)

| Property | Value |
|---|---|
| Device | Nihon Kohden EEG 2100 |
| Electrodes | 19 scalp (10–20 system) + A1/A2 on the mastoids for impedance check |
| Sampling rate | 500 Hz |
| Condition | Resting state, eyes closed, seated |
| Amplifier | Sensitivity 10 µV/mm, time constant 0.3 s, high-frequency filter 70 Hz |
| Recording montage included | Referential, **Cz** as common reference |

### 2.3 Parameters that are genuinely under-specified

A faithful reproduction requires values the sources do not give. We record these as
**uncertainties** rather than inventing them silently. The most consequential one:

> **The A1–A2 re-reference cannot be reproduced from the shared data.** The distributed
> raw files contain only the 19 scalp electrodes; A1 and A2 are described as reference
> electrodes used for impedance checking and are not present as data channels. A
> linked-mastoid re-reference is therefore not recomputable. We substitute an average
> reference (which is also what ICLabel was designed for) and apply the *same* choice in
> Pipeline B, so referencing is a controlled variable and cannot confound the comparison.

A second important one: the README says ICLabel rejected *"jaw artifacts"*, but ICLabel
has no such class — its seven classes are brain, muscle artifact, eye blink, heart beat,
line noise, channel noise, and other. We map *eye artifacts* → `eye blink` and
*jaw artifacts* → `muscle artifact`, and document the mapping.

In [ ]:
# The full uncertainty register, saved alongside the results.
unc = pd.DataFrame(ds.PIPELINE_A_UNCERTAINTIES)
ds.save_results(cfg, "pipeline_a_uncertainties", unc)

print(ds.NOTE_ON_REFERENCE)
print("=" * 88)
print("PIPELINE A UNCERTAINTY REGISTER")
print("=" * 88)
for i, row in unc.iterrows():
    print(f"\n[{i+1}] {row['parameter']}")
    print(f"    Issue      : {row['issue']}")
    print(f"    Resolution : {row['resolution']}")

---
## 3. Dataset acquisition

We download per subject rather than pulling the whole ~5 GB dataset, which keeps Colab's
disk within limits and makes the step restartable.

In [ ]:
# The dataset is downloaded at RUNTIME straight from OpenNeuro -- nothing needs to be
# in your Drive beforehand. This first call fetches ONLY the small top-level metadata
# (a few kB), because participants.tsv is what tells us who the subjects are and which
# diagnostic group each belongs to. Subject selection depends on it, so it must come
# first. The retries and the post-download verification live inside the function.
meta = ds.download_metadata(cfg)

print(f"Attempts: {meta['attempts']}")
if meta["ok"]:
    print("Metadata present:", ", ".join(ds.REQUIRED_METADATA))
else:
    print("MISSING:", meta["missing"])
    for e in meta["errors"]:
        print("  ", e)
    raise RuntimeError(
        "Could not fetch dataset metadata from OpenNeuro. Check network connectivity "
        "and that `openneuro-py` installed correctly, then re-run this cell."
    )

### 3.1 Select subjects (group-stratified)

Subject IDs in this dataset are ordered by diagnosis. Taking "the first N" would silently produce a single-class subset, so reduced-size runs stratify by group.

In [ ]:
# Subject IDs in this dataset are ordered by diagnosis, so naively taking "the first N"
# would return an all-Alzheimer's subset with no controls and make classification
# impossible. We therefore select a GROUP-STRATIFIED subset for reduced-size runs.
subjects = ds.select_subjects_balanced(cfg)

participants = pd.read_csv(Path(cfg.dataset_path) / "participants.tsv", sep="\t")
group_col = ds._find_group_column(participants)
groups = dict(zip(participants["participant_id"].astype(str),
                  participants[group_col].astype(str)))

sel_groups = pd.Series([groups.get(s, "?") for s in subjects]).value_counts().to_dict()
print(f"Selected {len(subjects)} subjects: {sel_groups}")
print(f"Group label meaning: {ds.GROUP_LABELS}")
print("First few:", subjects[:8])

### 3.2 Download the selected subjects' EEG

In [ ]:
# Now fetch the EEG for the selected subjects only (raw + the authors' derivatives).
# Roughly 60 MB per subject for both versions, so 20 subjects is about 1.2 GB.
#
# `ensure_dataset` is idempotent: it checks what is already on disk and downloads only
# what is missing. After a Colab timeout you can simply re-run this cell and it will
# pick up where it left off rather than starting over.
report = ds.ensure_dataset(cfg, subjects, include_derivatives=True)

if report.get("already_present"):
    print(f"Already on disk : {len(report['already_present'])} subjects (skipped)")
print(f"Newly downloaded: {len(report['downloaded'])} subjects")

if not report["ok"]:
    print()
    print("No EEG file found for:", report["missing"])
    for e in report["errors"]:
        print("  ", e)
    raise RuntimeError(
        "Some subjects failed to download. Re-run this cell to retry just those "
        "(already-downloaded subjects are skipped), or reduce MAX_SUBJECTS."
    )

print()
print("On-disk dataset size:", ds.directory_size_mb(Path(cfg.dataset_path)), "MB")

---
## 4. Dataset inventory and integrity

Requirement: build an inventory and check for missing recordings, inconsistent sampling
rates, unexpected channels, malformed files and duplicate subjects — **before** processing.

In [ ]:
# Dataset inventory: one row per subject with channel count, sampling rate, duration,
# file size and any load errors. The raw dataset is never modified.
inventory = ds.build_inventory(cfg, subjects=subjects)
ds.save_results(cfg, "subject_metadata", inventory)

cols = ["participant_id", "group", "n_channels", "sfreq_hz", "duration_s",
        "raw_size_mb", "derivative_exists", "derivative_duration_s"]
display(inventory[cols].head(12))
print(f"\nInventory saved with {len(inventory)} rows.")

### 4.1 Integrity checks

In [ ]:
# Integrity checks required before any processing.
integrity = ds.check_inventory_integrity(inventory)
ds.save_results(cfg, "dataset_integrity", integrity)

for k, v in integrity.items():
    print(f"{k:<38} {v}")

if integrity["inconsistent_sampling_rate"]:
    print("\nWARNING: sampling rates differ across subjects -- resampling is mandatory.")
if integrity["subjects_with_load_errors"]:
    print("\nWARNING: some files failed to load; they will be reported as exclusions.")

### 4.2 Verify the headline dataset facts independently

The project brief supplied numbers (88 subjects, 36/23/29, 19 channels, 500 Hz). We check them against the files themselves rather than assuming they are right.

In [ ]:
# Verify against the FULL participants.tsv, not just our working subset.
full = pd.read_csv(Path(cfg.dataset_path) / "participants.tsv", sep="\t")
gcol = ds._find_group_column(full)
observed = full[gcol].value_counts().to_dict()

claimed = {"A": 36, "F": 23, "C": 29}
print(f"Total subjects in participants.tsv : {len(full)}   (brief claimed 88)")
print(f"Observed group counts              : {observed}")
print(f"Brief claimed                      : {claimed} (AD / FTD / CN)")

checks = {
    "n_subjects_matches_88": len(full) == 88,
    "group_counts_match": {k: observed.get(k) for k in claimed} == claimed,
}
if inventory["n_channels"].notna().any():
    checks["all_19_channels"] = bool((inventory["n_channels"].dropna() == 19).all())
    checks["all_500_hz"] = bool((inventory["sfreq_hz"].dropna() == 500.0).all())

print("\nVerification results:")
for k, v in checks.items():
    print(f"  {'PASS' if v else 'MISMATCH'}  {k}")

ds.save_results(cfg, "dataset_verification",
                {"observed_groups": observed, "claimed_groups": claimed,
                 "n_subjects": int(len(full)), "checks": checks})

### 4.3 Dataset figures

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# -- class distribution --
counts = inventory["group"].value_counts().sort_index()
labels = [f"{g}\n({ds.GROUP_LABELS.get(g, g)})" for g in counts.index]
axes[0].bar(labels, counts.values, color=["#c44e52", "#4c72b0", "#55a868"][:len(counts)])
axes[0].set_ylabel("Number of subjects")
axes[0].set_xlabel("Diagnostic group")
axes[0].set_title(f"Class distribution (n = {int(counts.sum())} subjects)")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.1, str(v), ha="center", fontweight="bold")

# -- recording duration --
dur = inventory["duration_s"].dropna() / 60.0
axes[1].hist(dur, bins=15, color="#4c72b0", edgecolor="black", alpha=0.8)
axes[1].axvline(dur.mean(), color="red", ls="--",
                label=f"mean = {dur.mean():.1f} min")
axes[1].set_xlabel("Recording duration (minutes)")
axes[1].set_ylabel("Number of subjects")
axes[1].set_title("Distribution of recording durations")
axes[1].legend()

plt.tight_layout()
plt.savefig(cfg.figures_dir / "dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print("Interpretation: this is the sample actually available to the classifier. "
      "Class sizes set the ceiling on how precisely any accuracy can be estimated.")

---
## 5. Pipeline A on a single subject

Before batch-processing anything, we run one subject and look at the result. Silent
batch jobs are how bad preprocessing gets published.

**Pipeline A steps:**

```
Raw EEG (19 ch, 500 Hz)
  → Butterworth band-pass 0.5–45 Hz (zero-phase)
  → Re-reference           (documented substitution — see §2.3)
  → Trim onset + resample to 100 Hz   [shared with Pipeline B]
  → ASR (0.5 s window, cutoff SD = 17)
  → Extended-Infomax ICA (RunICA equivalent)
  → ICLabel → reject eye-blink + muscle components
  → cleaned EEG
```

In [ ]:
demo_sub = subjects[0]
print(f"Demonstration subject: {demo_sub} (group {groups[demo_sub]})")

raw_demo = ds.load_raw_subject(cfg, demo_sub, preload=True)
print(f"Raw : {len(raw_demo.ch_names)} channels, {raw_demo.info['sfreq']:.0f} Hz, "
      f"{raw_demo.n_times / raw_demo.info['sfreq']:.1f} s")
print(f"Channels: {raw_demo.ch_names}")

raw_before = raw_demo.copy()          # keep an untouched copy for the before/after figure
clean_A, meta_A = ds.pipeline_a(raw_demo, cfg)

print(f"\nCleaned: {clean_A.info['sfreq']:.0f} Hz, {meta_A['output_duration_s']:.1f} s")
print(f"ICA components rejected: {meta_A.get('n_components_rejected')}")
print(f"Total preprocessing wall time: {meta_A['total_wall_time_s']:.2f} s")
if meta_A["warnings"]:
    print("\nWARNINGS:")
    for w in meta_A["warnings"]:
        print("  -", w)

### 5.1 Per-step cost breakdown

This is where Pipeline A's computational profile becomes visible, and it motivates the choice of alternative in Notebook 02.

In [ ]:
steps = pd.DataFrame(meta_A["steps"]).T[["wall_time_s", "cpu_time_s", "peak_rss_mb"]]
steps["pct_of_total"] = (steps["wall_time_s"] / steps["wall_time_s"].sum() * 100).round(1)
display(steps)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(steps.index, steps["wall_time_s"], color="#4c72b0", edgecolor="black")
ax.set_xlabel("Wall-clock time (seconds)")
ax.set_ylabel("Pipeline A step")
ax.set_title(f"Pipeline A cost per step — subject {demo_sub}")
for i, v in enumerate(steps["wall_time_s"]):
    ax.text(v, i, f" {v:.1f}s", va="center")
plt.tight_layout()
plt.savefig(cfg.figures_dir / "pipelineA_step_times.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Raw versus cleaned EEG

In [ ]:
def plot_before_after(raw_b, raw_a, title_b, title_a, seconds=10, n_ch=8, fname=None):
    """Plot the same time window before and after cleaning, on a shared amplitude scale."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    for ax, r, ttl in zip(axes, [raw_b, raw_a], [title_b, title_a]):
        sf = r.info["sfreq"]
        n = int(seconds * sf)
        # Start 30 s in to skip any onset transient.
        start = min(int(30 * sf), max(0, r.n_times - n))
        data = r.get_data(start=start, stop=start + n)[:n_ch] * 1e6   # -> microvolts
        t = np.arange(data.shape[1]) / sf
        offset = 120
        for i in range(data.shape[0]):
            ax.plot(t, data[i] + i * offset, lw=0.6, color="#2b2b2b")
        ax.set_yticks([i * offset for i in range(data.shape[0])])
        ax.set_yticklabels(r.ch_names[:n_ch])
        ax.set_xlabel("Time (s)")
        ax.set_title(ttl)
        ax.grid(alpha=0.25)
    axes[0].set_ylabel("Channel (traces offset by 120 µV for legibility)")
    plt.tight_layout()
    if fname:
        plt.savefig(cfg.figures_dir / fname, dpi=150, bbox_inches="tight")
    plt.show()

plot_before_after(raw_before, clean_A,
                  f"Raw EEG — {demo_sub}", f"After Pipeline A — {demo_sub}",
                  fname="example_raw_vs_pipelineA.png")
print("Interpretation: high-amplitude transients present in the raw traces should be "
      "attenuated after ASR and ICA component rejection. Genuine oscillatory activity "
      "should be preserved.")

---
## 6. Validating our reproduction against the authors' derivative

This is the part that distinguishes a reproduction from a re-implementation that merely
*looks* plausible. We compare:

- **A** = raw → *our* implementation of the authors' pipeline
- **B** = the authors' own supplied derivative file

**We should not expect identical signals.** EEGLAB's `runica` uses a different random
initialisation and convergence path from MNE's extended Infomax; ASR's eigenspace solvers
differ between MATLAB and Python; and the reference substitution described in §2.3 is a
genuine, unavoidable difference. The question is whether the two agree in the properties
that matter — spectral content and amplitude distribution.

A further complication: **ASR can remove data segments**, so the two recordings may differ
in length and be time-shifted. Sample-wise correlation on the overlapping prefix is
therefore reported as a *lower bound*; the length-robust spectral comparisons are the
primary evidence.

In [ ]:
try:
    der_demo = ds.load_derivative_subject(cfg, demo_sub, preload=True)
    print(f"Authors' derivative: {len(der_demo.ch_names)} channels, "
          f"{der_demo.info['sfreq']:.0f} Hz, "
          f"{der_demo.n_times / der_demo.info['sfreq']:.1f} s")

    comparison = ds.compare_signals(clean_A, der_demo, cfg,
                                    label=f"{demo_sub}_ourA_vs_authors_derivative")
    ds.save_results(cfg, f"validation_{demo_sub}", comparison)

    print(f"\nDuration  ours {comparison['duration_ours_s']:.1f} s  vs  "
          f"authors {comparison['duration_reference_s']:.1f} s  "
          f"(ratio {comparison['duration_ratio']:.3f})")
    print(f"Mean sample-wise Pearson r (LOWER BOUND): {comparison['pearson_r_mean']:.3f}")
    print(f"Mean log-PSD correlation                : {comparison['log_psd_correlation_mean']:.3f}")
    print(f"Min  log-PSD correlation                : {comparison['log_psd_correlation_min']:.3f}")
except FileNotFoundError as exc:
    comparison = None
    print(f"Derivative not available: {exc}")
    print("Validation against the authors' derivative: NOT COMPUTED (file missing).")

### 6.1 Where do the two differ, and by how much?

In [ ]:
if comparison is not None:
    keys = ["rms_v", "variance_v2", "spectral_entropy_mean", "sef95_mean_hz",
            "relpow_delta", "relpow_theta", "relpow_alpha", "relpow_beta",
            "frac_samples_robust_z_gt5"]
    rows = []
    for k in keys:
        d = comparison["metric_differences"].get(k)
        if d:
            rows.append({"metric": k, "our_pipeline_A": d["ours"],
                         "authors_derivative": d["reference"],
                         "abs_diff": d["abs_diff"],
                         "rel_diff_pct": round(d["rel_diff_pct"], 2)})
    val_table = pd.DataFrame(rows)
    ds.save_results(cfg, f"validation_table_{demo_sub}", val_table)
    display(val_table)
    print("\nRelative band powers are the most informative row group: they are "
          "amplitude-normalised, so they compare spectral SHAPE rather than the overall "
          "scale that the reference substitution necessarily changes.")
else:
    print("Not computed because the authors' derivative file was not available.")

### 6.2 Power spectral density comparison

In [ ]:
import scipy.signal as sps

def psd_of(raw, label):
    sf = raw.info["sfreq"]
    nfft = int(min(4 * sf, raw.n_times))
    f, p = sps.welch(raw.get_data(), fs=sf, nperseg=nfft, noverlap=nfft // 2, axis=-1)
    return f, p.mean(axis=0), label

if comparison is not None:
    fig, ax = plt.subplots(figsize=(9, 5))
    for raw_obj, lab, style in [(raw_before, "Raw (unprocessed)", ":"),
                                (clean_A, "Our Pipeline A", "-"),
                                (der_demo, "Authors' derivative", "--")]:
        f, p, lab = psd_of(raw_obj, lab)
        m = (f >= 0.5) & (f <= 45)
        ax.semilogy(f[m], p[m], style, lw=1.8, label=lab)

    for name, (lo, hi) in ds.FREQ_BANDS.items():
        ax.axvline(lo, color="grey", lw=0.5, alpha=0.5)
        ax.text(lo + 0.2, ax.get_ylim()[1] * 0.5, name, fontsize=7,
                rotation=90, color="grey", va="top")

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power spectral density (V²/Hz)")
    ax.set_title(f"PSD comparison — subject {demo_sub} (channel-averaged)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "psd_validation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Interpretation: agreement in spectral SHAPE between our Pipeline A and the "
          "authors' derivative is the evidence that the reproduction is faithful where "
          "it matters. Vertical offset reflects the referencing/scale difference and is "
          "expected.")
else:
    print("PSD validation figure: not produced (derivative unavailable).")

---
## 7. Batch processing

Each subject is processed independently and its features cached to Drive. If the Colab
session dies, re-running this cell resumes from where it stopped — already-cached
subjects are skipped, not recomputed.

Failures are caught per subject, logged with the subject ID and the error, and reported
in the exclusion table. No subject disappears silently.

In [ ]:
del raw_before, clean_A, raw_demo
try:
    del der_demo
except NameError:
    pass
import gc; gc.collect()

status_A = ds.process_many_subjects(cfg, subjects, pipeline="A", groups=groups)
ds.save_results(cfg, "processing_status_A", status_A)

n_ok = int((status_A["status"].isin(["ok", "cached"])).sum())
print(f"\nSucceeded: {n_ok}/{len(subjects)}")

### 7.1 Exclusion report

Requirement: report subjects expected, processed, failed and excluded — with reasons.

In [ ]:
ok_mask = status_A["status"].isin(["ok", "cached"])
failed = status_A.loc[~ok_mask, ["participant_id", "group", "status", "error"]]

report_A = {
    "total_subjects_expected": len(subjects),
    "successfully_processed": int(ok_mask.sum()),
    "failed": int((~ok_mask).sum()),
    "failed_subject_ids": failed["participant_id"].tolist(),
    "failure_reasons": dict(zip(failed["participant_id"], failed["error"])),
}
ds.save_results(cfg, "exclusion_report_A", report_A)

print(f"Expected              : {report_A['total_subjects_expected']}")
print(f"Successfully processed: {report_A['successfully_processed']}")
print(f"Failed                : {report_A['failed']}")
if report_A["failed"]:
    print("\nFailures (subject -> reason):")
    for sid, err in report_A["failure_reasons"].items():
        print(f"  {sid}: {err}")
else:
    print("\nNo failures.")

subjects_ok = status_A.loc[ok_mask, "participant_id"].tolist()

### 7.2 Signal-quality results

Each metric is a standard, individually interpretable quantity. We deliberately avoid inventing a single opaque "signal quality score".

In [ ]:
rows = []
for _, r in status_A[ok_mask].iterrows():
    sid = r["participant_id"]
    meta_path = cfg.cache_dir / "pipeline_A" / f"{sid}_meta.json"
    if not meta_path.exists():
        continue
    with open(meta_path) as fh:
        m = json.load(fh)
    before, after = m.get("quality_before", {}), m.get("quality_after", {})
    if not after:
        continue
    row = {"participant_id": sid, "group": m.get("group"), "pipeline": "A"}
    for k, v in after.items():
        if isinstance(v, (int, float)):
            row[f"after_{k}"] = v
            if isinstance(before.get(k), (int, float)):
                row[f"before_{k}"] = before[k]
    row["n_components_rejected"] = m.get("preprocess_meta", {}).get("n_components_rejected")
    rows.append(row)

quality_A = pd.DataFrame(rows)
ds.save_results(cfg, "signal_quality_results_A", quality_A)

show = [c for c in ["participant_id", "group", "before_rms_v", "after_rms_v",
                    "before_frac_samples_robust_z_gt5", "after_frac_samples_robust_z_gt5",
                    "after_relpow_alpha", "n_components_rejected"] if c in quality_A.columns]
display(quality_A[show].head(12))
print(f"\nSignal-quality table: {quality_A.shape[0]} subjects x {quality_A.shape[1]} columns")

---
## 8. Downstream analysis

Everything from here on is **shared code**, called identically by Notebook 02. That is
what makes the eventual A-vs-B difference attributable to preprocessing alone.

### 8.1 Feature set

Per epoch, per channel (19 channels × 9 features = 171 features):

| Feature | Count | Rationale |
|---|---|---|
| Relative band power (δ, θ, α, β, low-γ) | 5 × 19 | Spectral slowing is the most reproducible qEEG marker of dementia |
| Spectral entropy | 1 × 19 | Captures spectral flattening not visible in band ratios |
| 95% spectral edge frequency | 1 × 19 | Single-number summary of the spectral shift |
| Hjorth mobility, complexity | 2 × 19 | Cheap time-domain descriptors of signal dynamics |

**Why *relative* rather than absolute band power?** Absolute amplitude is directly altered
by preprocessing itself — ASR and ICA *remove variance by construction*. Absolute power
would therefore partly measure "how much did this pipeline subtract", which is exactly the
confound we must avoid when preprocessing is the independent variable. Relative power
normalises that out.

In [ ]:
features_A = ds.assemble_feature_table(cfg, "A", subjects_ok)
feature_names = [c for c in features_A.columns
                 if c not in ("participant_id", "group", "epoch_index")]

ds.save_results(cfg, "feature_names", {"feature_names": feature_names,
                                       "n_features": len(feature_names)})

print(f"Feature matrix : {features_A.shape[0]} epochs x {len(feature_names)} features")
print(f"Subjects       : {features_A['participant_id'].nunique()}")
print(f"Epochs/subject : median {features_A.groupby('participant_id').size().median():.0f}")
print(f"\nEpochs per group:\n{features_A.groupby('group').size()}")
print(f"\nSubjects per group:\n{features_A.groupby('group')['participant_id'].nunique()}")

### 8.2 Leakage prevention

This dataset has many epochs per subject. Splitting epochs at random would place epochs
from the *same person* in both training and test sets, and the model would learn that
person's electrode impedance and baseline rhythm rather than their pathology. The AHEPA
benchmark review quantifies exactly this: studies using epoch-level *k*-fold report
accuracies roughly 7–10 percentage points higher than subject-level designs.

Three guarantees, all enforced inside `run_cross_validation`:

1. **Subject-level splitting.** `StratifiedGroupKFold` with `groups = participant_id`; every epoch of a subject lands wholly in train or wholly in test. An explicit assertion raises if any subject ever appears on both sides.
2. **Scaling inside the fold.** `StandardScaler` sits inside the sklearn `Pipeline`, so it is fitted on training epochs only.
3. **No selection on full data.** No feature selection or hyper-parameter search is performed outside a fold.

Headline metrics are computed **after aggregating epoch predictions to the subject**,
because the subject — not the 4-second epoch — is the clinical unit of interest.

In [ ]:
# Live leakage check on the actual splits this configuration will generate.
from sklearn.model_selection import StratifiedGroupKFold

sub_ad_cn = features_A[features_A["group"].isin(["A", "C"])]
gg = sub_ad_cn["participant_id"].to_numpy()
yy = sub_ad_cn["group"].to_numpy()

n_per_class = sub_ad_cn.groupby("group")["participant_id"].nunique()
n_splits = min(cfg.cv_n_splits, int(n_per_class.min()))
print(f"Subjects per class: {n_per_class.to_dict()}  ->  using {n_splits} folds")

violations = 0
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed)
for tr, te in sgkf.split(np.zeros(len(yy)), yy, gg):
    if set(gg[tr]) & set(gg[te]):
        violations += 1

print(f"Folds checked           : {n_splits}")
print(f"Subject-overlap failures: {violations}")
print("PASS - no subject appears in both training and test." if violations == 0
      else "FAIL - leakage detected!")

### 8.3 Classification tasks

| Task | Comparison | Included because |
|---|---|---|
| 1 | AD vs CN | The field's primary benchmark; largest groups |
| 2 | AD vs FTD | Clinically the hardest and most useful differential |
| 3 | AD vs FTD vs CN | Three-class; reported with the caveat that FTD has only 23 subjects |

**Validation strategy.** Primary: repeated stratified group *k*-fold (10 folds × 5
repeats), because it enforces subject independence *and* averages over many partitions,
avoiding the split-dependent instability of a single hold-out. Secondary:
leave-one-subject-out, which is the convention in the most rigorous studies on this
dataset and lets us compare against published baselines.

**Reference baselines** (from the AHEPA benchmark review, restricted to the most rigorous
"Validity 1" studies): AD vs CN ≈ 82% accuracy, AD vs FTD ≈ 71%, three-class ≈ 70%.
Results far above these should be treated as evidence of a methodological problem, not
of success.

In [ ]:
TASKS = {
    "AD_vs_CN":        ["A", "C"],
    "AD_vs_FTD":       ["A", "F"],
    "AD_vs_FTD_vs_CN": ["A", "F", "C"],
}

results_A = {}
for task_name, classes in TASKS.items():
    print(f"\n{'=' * 70}\nTASK: {task_name}   classes={classes}\n{'=' * 70}")
    try:
        res = ds.run_cross_validation(
            features_A, feature_names, classes, cfg,
            classifier=cfg.primary_classifier, scheme="sgkf", pipeline_label="A",
        )
        if "error" in res:
            print("SKIPPED:", res["error"])
            results_A[task_name] = res
            continue
        s = res["summary"]
        print(f"Subjects: {res['n_subjects']}  {res['subjects_per_class']}")
        print(f"Folds   : {res['n_splits_total']} ({res.get('effective_n_splits')} x {cfg.cv_n_repeats} repeats)")
        print(f"  Balanced accuracy : {s['balanced_accuracy_mean']:.3f} (SD {s['balanced_accuracy_std']:.3f})")
        print(f"  Accuracy          : {s['accuracy_mean']:.3f}")
        print(f"  Macro F1          : {s['f1_macro_mean']:.3f}")
        if "roc_auc_mean" in s:
            print(f"  ROC-AUC           : {s['roc_auc_mean']:.3f}")
            print(f"  Sensitivity       : {s['sensitivity_mean']:.3f}")
            print(f"  Specificity       : {s['specificity_mean']:.3f}")
        results_A[task_name] = res
    except Exception as exc:
        print(f"FAILED: {exc!r}")
        results_A[task_name] = {"error": repr(exc)}

ds.save_results(cfg, "classification_results_A", results_A)

### 8.4 Per-fold results table

Saved so Notebook 03 can perform paired fold-level comparisons.

In [ ]:
fold_rows = []
for task_name, res in results_A.items():
    for fr in res.get("fold_results", []):
        fold_rows.append({"pipeline": "A", "task": task_name, **fr})

fold_results_A = pd.DataFrame(fold_rows)
if not fold_results_A.empty:
    fold_results_A = fold_results_A.drop(
        columns=[c for c in ["confusion_matrix", "confusion_matrix_labels"]
                 if c in fold_results_A.columns])
ds.save_results(cfg, "fold_results_A", fold_results_A)

if not fold_results_A.empty:
    display(fold_results_A.groupby("task")[
        ["balanced_accuracy", "f1_macro", "accuracy"]].agg(["mean", "std"]).round(3))
else:
    print("No fold results produced -- check the messages above.")

### 8.5 Out-of-fold subject predictions

These are the raw material for the paired statistical tests in Notebook 03. Because the same subjects are scored by both pipelines, the comparison can be genuinely paired.

In [ ]:
oof_rows = []
for task_name, res in results_A.items():
    for r in res.get("oof_predictions", []):
        oof_rows.append({"pipeline": "A", "task": task_name, **r})

oof_A = pd.DataFrame(oof_rows)
ds.save_results(cfg, "oof_predictions_A", oof_A)
print(f"Out-of-fold predictions: {oof_A.shape}")
display(oof_A.head(8))

### 8.6 Secondary validation — leave-one-subject-out

LOSO is the convention in the most rigorous published studies on this dataset, so running it lets us compare like with like. It is more expensive, hence secondary.

In [ ]:
loso_A = {}
if cfg.run_loso:
    for task_name, classes in TASKS.items():
        try:
            res = ds.run_cross_validation(features_A, feature_names, classes, cfg,
                                          classifier=cfg.primary_classifier,
                                          scheme="loso", pipeline_label="A")
            if "error" in res:
                print(f"{task_name:<18} skipped: {res['error']}")
                continue
            p = res["pooled_subject_level"]
            print(f"{task_name:<18} bal-acc {p['balanced_accuracy']:.3f}  "
                  f"macro-F1 {p['f1_macro']:.3f}  (n={res['n_subjects']})")
            loso_A[task_name] = res
        except Exception as exc:
            print(f"{task_name:<18} FAILED: {exc!r}")
    ds.save_results(cfg, "loso_results_A", loso_A)
else:
    print("LOSO disabled in configuration.")

### 8.7 Robustness — secondary classifier

A second, structurally different model checks that any conclusion is not an artefact of one classifier's inductive bias.

In [ ]:
results_A_rf = {}
for task_name, classes in TASKS.items():
    try:
        res = ds.run_cross_validation(features_A, feature_names, classes, cfg,
                                      classifier=cfg.secondary_classifier,
                                      scheme="sgkf", pipeline_label="A")
        if "error" in res:
            print(f"{task_name:<18} skipped: {res['error']}")
            continue
        s = res["summary"]
        print(f"{task_name:<18} bal-acc {s['balanced_accuracy_mean']:.3f} "
              f"(SD {s['balanced_accuracy_std']:.3f})")
        results_A_rf[task_name] = res
    except Exception as exc:
        print(f"{task_name:<18} FAILED: {exc!r}")

ds.save_results(cfg, "classification_results_A_rf", results_A_rf)

### 8.8 Comparison against the published literature

Requirement #24: attempt to reproduce published performance and explain differences.

An important caveat before reading this table: the published figures come from *different
feature sets, classifiers and subject subsets*. They are a sanity range, not a target. Our
result being lower is not necessarily a failure, and our result being much higher would be
a red flag.

In [ ]:
# Benchmark values from the AHEPA scoping review (Miltiadous et al., 2026,
# Cogn Neurodyn 20:95), restricted to "Validity 1" studies -- those using subject-level
# validation with no leakage. These are the only literature numbers comparable to ours.
literature = {
    "AD_vs_CN":        {"published_validity1_accuracy": 0.8211, "source": "AHEPA benchmark, Validity-1 mean"},
    "AD_vs_FTD":       {"published_validity1_accuracy": 0.7144, "source": "AHEPA benchmark, Validity-1 mean"},
    "AD_vs_FTD_vs_CN": {"published_validity1_accuracy": 0.6999, "source": "AHEPA benchmark, Validity-1 mean"},
}

rows = []
for task_name, lit in literature.items():
    res = results_A.get(task_name, {})
    ours = res.get("summary", {}).get("accuracy_mean")
    rows.append({
        "task": task_name,
        "published_validity1_accuracy": lit["published_validity1_accuracy"],
        "our_reproduction_accuracy": round(ours, 4) if ours is not None else "not computed",
        "difference": round(ours - lit["published_validity1_accuracy"], 4) if ours is not None else "n/a",
        "source": lit["source"],
    })

lit_table = pd.DataFrame(rows)
ds.save_results(cfg, "literature_comparison_A", lit_table)
display(lit_table)

print("""
Expected causes of any difference:
  - different feature set (we use spectral + Hjorth; published studies vary widely)
  - different classifier (we use regularised logistic regression as a transparent baseline)
  - subject subset (reduced run modes use fewer subjects than the full 88)
  - the reference substitution documented in section 2.3
  - MNE/Python vs EEGLAB/MATLAB implementations of ASR and ICA
  - unavailable original code, hyper-parameters and random seeds
We do NOT tune anything to close this gap; doing so would invalidate the comparison.
""")

### 8.9 Confusion matrices and ROC curves

In [ ]:
from sklearn.metrics import roc_curve, auc

valid_tasks = [t for t, r in results_A.items() if "pooled_subject_level" in r]
if valid_tasks:
    fig, axes = plt.subplots(1, len(valid_tasks), figsize=(5 * len(valid_tasks), 4.2))
    axes = np.atleast_1d(axes)
    for ax, task_name in zip(axes, valid_tasks):
        p = results_A[task_name]["pooled_subject_level"]
        cm = np.array(p["confusion_matrix"])
        labs = p["confusion_matrix_labels"]
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(labs)), labs)
        ax.set_yticks(range(len(labs)), labs)
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title(f"{task_name}\nbal-acc = {p['balanced_accuracy']:.3f}")
        thr = cm.max() / 2 if cm.max() else 0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > thr else "black", fontweight="bold")
        ax.grid(False)
    plt.suptitle("Pipeline A — subject-level confusion matrices (counts = subjects)", y=1.03)
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "confusion_matrices_A.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No completed tasks to plot.")

In [ ]:
# ROC curves for the two binary tasks (undefined for the three-class problem).
binary_tasks = [t for t in ["AD_vs_CN", "AD_vs_FTD"]
                if t in results_A and "oof_predictions" in results_A[t]]
if binary_tasks:
    fig, ax = plt.subplots(figsize=(6.2, 5.6))
    for task_name in binary_tasks:
        oof = pd.DataFrame(results_A[task_name]["oof_predictions"])
        classes = sorted(TASKS[task_name])
        pos = classes[1]
        y_true = (oof["true_class"] == pos).astype(int)
        y_score = oof[f"proba_{pos}"]
        if y_true.nunique() < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true, y_score)
        ax.plot(fpr, tpr, lw=2, label=f"{task_name} (AUC = {auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
    ax.set_xlabel("False positive rate (1 − specificity)")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_title("Pipeline A — subject-level ROC curves")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "roc_curves_A.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No binary task results available for ROC curves.")

---
## 9. Computational benchmarking

Requirement #16. All timings below come from the single Colab environment recorded in
§1.6; cross-environment comparison would be meaningless.

In [ ]:
rows = []
for _, r in status_A[ok_mask].iterrows():
    sid = r["participant_id"]
    mp = cfg.cache_dir / "pipeline_A" / f"{sid}_meta.json"
    if not mp.exists():
        continue
    with open(mp) as fh:
        m = json.load(fh)
    if m.get("status") not in ("ok", "cached"):
        continue
    pm = m.get("preprocess_meta", {})
    steps = pm.get("steps", {})
    rows.append({
        "participant_id": sid, "group": m.get("group"), "pipeline": "A",
        "input_duration_s": m.get("input_duration_s"),
        "load_s": m.get("timing_load", {}).get("wall_time_s"),
        "preprocess_s": m.get("timing_preprocess", {}).get("wall_time_s"),
        "epoch_s": m.get("timing_epoch", {}).get("wall_time_s"),
        "features_s": m.get("timing_features", {}).get("wall_time_s"),
        "total_s": m.get("timing_total", {}).get("wall_time_s"),
        "peak_rss_mb": m.get("timing_total", {}).get("peak_rss_mb"),
        "asr_s": steps.get("asr", {}).get("wall_time_s"),
        "ica_s": steps.get("ica_iclabel", {}).get("wall_time_s"),
        "filter_s": steps.get("filter", {}).get("wall_time_s"),
        "n_epochs": m.get("n_epochs"),
    })

runtime_A = pd.DataFrame(rows)
if not runtime_A.empty:
    runtime_A["s_per_min_recording"] = (
        runtime_A["total_s"] / (runtime_A["input_duration_s"] / 60.0))
ds.save_results(cfg, "runtime_results_A", runtime_A)

if not runtime_A.empty:
    display(runtime_A[["total_s", "preprocess_s", "asr_s", "ica_s", "features_s",
                       "peak_rss_mb", "s_per_min_recording"]].describe().round(2))
    print(f"\nMean total time per subject : {runtime_A['total_s'].mean():.1f} s")
    print(f"Total for {len(runtime_A)} subjects  : {runtime_A['total_s'].sum() / 60:.1f} min")
    if runtime_A["ica_s"].notna().any():
        share = runtime_A["ica_s"].sum() / runtime_A["total_s"].sum() * 100
        print(f"ICA + ICLabel share of runtime: {share:.1f}%")
    if runtime_A["asr_s"].notna().any():
        share = runtime_A["asr_s"].sum() / runtime_A["total_s"].sum() * 100
        print(f"ASR share of runtime          : {share:.1f}%")
else:
    print("No runtime records -- no subjects completed.")

### 9.1 Storage cost

In [ ]:
storage_A = {
    "cache_pipeline_A_mb": ds.directory_size_mb(cfg.cache_dir / "pipeline_A"),
    "results_mb": ds.directory_size_mb(cfg.results_dir),
    "figures_mb": ds.directory_size_mb(cfg.figures_dir),
    "dataset_on_disk_mb": ds.directory_size_mb(Path(cfg.dataset_path)),
    "n_subjects_cached": len(subjects_ok),
}
if subjects_ok:
    storage_A["mb_per_subject"] = round(
        storage_A["cache_pipeline_A_mb"] / len(subjects_ok), 3)
ds.save_results(cfg, "storage_A", storage_A)
for k, v in storage_A.items():
    print(f"{k:<28} {v}")

### 9.2 Scalability

How does runtime grow with the number of subjects? We derive this from the per-subject
timings we already measured, rather than re-processing the data at several sample sizes —
re-running the pipeline five times over would waste Colab quota to answer a question the
existing measurements already contain.

In [ ]:
if not runtime_A.empty and len(runtime_A) >= 3:
    times = runtime_A["total_s"].to_numpy()
    n_max = len(times)
    checkpoints = [n for n in [5, 10, 20, 40, 88] if n <= n_max]
    if n_max not in checkpoints:
        checkpoints.append(n_max)

    rows = []
    cumulative = np.cumsum(times)
    for n in checkpoints:
        rows.append({
            "n_subjects": n,
            "cumulative_runtime_s": round(cumulative[n - 1], 1),
            "cumulative_runtime_min": round(cumulative[n - 1] / 60, 2),
            "mean_runtime_per_subject_s": round(cumulative[n - 1] / n, 2),
            "peak_rss_mb": round(runtime_A["peak_rss_mb"].iloc[:n].max(), 1),
            "measured": True,
        })
    scal_A = pd.DataFrame(rows)

    # Linear fit -- preprocessing is per-subject independent, so cost should be O(n).
    coef = np.polyfit(scal_A["n_subjects"], scal_A["cumulative_runtime_s"], 1)
    scal_A.attrs["slope_s_per_subject"] = float(coef[0])

    est_88 = float(np.polyval(coef, 88))
    print(f"Fitted slope       : {coef[0]:.2f} s per additional subject")
    print(f"Extrapolated to 88 : {est_88 / 60:.1f} min "
          f"({'MEASURED' if n_max >= 88 else 'ESTIMATE - beyond measured range'})")

    if n_max < 88:
        scal_A = pd.concat([scal_A, pd.DataFrame([{
            "n_subjects": 88,
            "cumulative_runtime_s": round(est_88, 1),
            "cumulative_runtime_min": round(est_88 / 60, 2),
            "mean_runtime_per_subject_s": round(est_88 / 88, 2),
            "peak_rss_mb": np.nan,
            "measured": False,      # explicitly flagged as extrapolation
        }])], ignore_index=True)

    ds.save_results(cfg, "scalability_A", scal_A)
    display(scal_A)

    fig, ax = plt.subplots(figsize=(7.5, 4.4))
    m = scal_A["measured"]
    ax.plot(scal_A.loc[m, "n_subjects"], scal_A.loc[m, "cumulative_runtime_min"],
            "o-", lw=2, label="Measured")
    if (~m).any():
        ax.plot(scal_A.loc[~m, "n_subjects"], scal_A.loc[~m, "cumulative_runtime_min"],
                "s--", lw=2, color="orange", label="Extrapolated (not measured)")
    ax.set_xlabel("Number of subjects processed")
    ax.set_ylabel("Cumulative runtime (minutes)")
    ax.set_title("Pipeline A — scalability")
    ax.legend()
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "scalability_A.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Scalability analysis: NOT COMPUTED (fewer than 3 subjects processed).")

---
## 10. Summary of Notebook 01

This cell prints only values that were actually computed in this session. Anything that
could not be computed is stated as such rather than filled in.

In [ ]:
summary = {
    "notebook": "01_author_pipeline_ds004504",
    "pipeline": "A (reproduction of dataset authors' method)",
    "run_mode": cfg.mode(),
    "subjects_requested": len(subjects),
    "subjects_processed": len(subjects_ok),
    "subjects_failed": int((~ok_mask).sum()),
    "n_features": len(feature_names),
    "n_epochs_total": int(len(features_A)),
    "random_seed": cfg.random_seed,
}
for task_name, res in results_A.items():
    s = res.get("summary", {})
    summary[f"{task_name}_balanced_accuracy"] = (
        round(s["balanced_accuracy_mean"], 4) if "balanced_accuracy_mean" in s
        else "not computed")
if not runtime_A.empty:
    summary["mean_runtime_per_subject_s"] = round(runtime_A["total_s"].mean(), 2)
    summary["mean_peak_rss_mb"] = round(runtime_A["peak_rss_mb"].mean(), 1)

ds.save_results(cfg, "summary_A", summary)

print("=" * 70)
print("NOTEBOOK 01 SUMMARY - Pipeline A")
print("=" * 70)
for k, v in summary.items():
    print(f"{k:<38} {v}")
print("\nAll results written to:", cfg.results_dir)
print("\nNEXT: run 02_alternative_pipeline_ds004504.ipynb")

---
## 11. Notes and limitations of this notebook

**What was demonstrated here** (measured in this session):
- Pipeline A reproduced from raw EEG, with its per-step computational cost.
- Signal-level agreement between our reproduction and the authors' derivative.
- Subject-level classification performance under a leakage-free protocol.

**What is *not* claimed:**
- That our reproduction is bit-identical to the authors' output. It cannot be: EEGLAB's
  `runica` seed and convergence path are unavailable, ASR's solvers differ between MATLAB
  and Python, and the A1–A2 reference is not recomputable from the shared files.
- That these numbers represent clinical diagnostic capability. This is a methodological
  experiment on 88 subjects from one hospital, not a clinical validation study.

**Carried forward to Notebook 03:** `classification_results_A`, `fold_results_A`,
`oof_predictions_A`, `runtime_results_A`, `signal_quality_results_A`, `scalability_A`.